# Spatial LLM: Embodied Spatiotemporal Language Model

This notebook trains and evaluates a Spatial LLM with gain-field coordinate transformations.

**Architecture:**
- Dual egocentric/allocentric streams fused via learned gating
- Gain-field transform for coordinate frame conversion
- Standard transformer backbone with NaN-safe training

**Cells:**
1. Setup & install dependencies
2. Configuration
3. Model verification
4. Generate synthetic data
5. Train on synthetic data
6. Train on real spatial datasets (SpartQA, bAbI, CLEVR)
7. Evaluation & results

## Cell 1: Setup

In [ ]:
import os
import sys
import shutil

# -- Option A: Running from a Kaggle dataset upload --
INPUT_DIR = '/kaggle/input/spatial-llm-project/Spatial-LLM'
WORK_DIR = '/kaggle/working/Spatial-LLM'
BRANCH = 'claude/refactor-kaggle-code-dG3li'

if os.path.exists(INPUT_DIR):
    if not os.path.exists(WORK_DIR):
        shutil.copytree(INPUT_DIR, WORK_DIR)
        print(f'Copied project to {WORK_DIR}')
    os.chdir(WORK_DIR)
elif os.path.exists('/kaggle/working'):
    # -- Option B: Clone from GitHub --
    os.chdir('/kaggle/working')
    if not os.path.exists('Spatial-LLM'):
        !git clone --branch {BRANCH} https://github.com/Mohammadzamanid/Spatial-LLM.git
    os.chdir('Spatial-LLM')
else:
    # -- Option C: Local run --
    pass

# Ensure src is importable from the project root
sys.path.insert(0, os.getcwd())
print(f'Working directory: {os.getcwd()}')
print(f'Contents: {sorted(os.listdir("."))}')

# Verify src/ exists
if os.path.isdir('src'):
    print(f'src/ found with: {sorted(os.listdir("src"))}')
else:
    print('WARNING: src/ directory not found! Check clone/copy.')

In [ ]:
!pip install -q pyyaml tqdm einops

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2: Configuration

In [ ]:
import yaml

CONFIG = {
    'model': {
        'vocab_size': 50257,
        'd_model': 256,
        'n_layers': 6,
        'n_heads': 4,
        'n_reference_frames': 4,
        'max_seq_len': 512,
        'dropout': 0.1,
        'n_schema_types': 8,
        'memory_slots': 256,
        'visual_dim': 512,
    },
    'training': {
        'batch_size': 8,
        'learning_rate': 5e-5,
        'weight_decay': 0.01,
        'num_epochs': 10,
        'gradient_clip': 0.5,
        'warmup_steps': 200,
        'scheduler_type': 'cosine',
        'aux_loss_weight': 0.01,
        'temporal_smoothness_weight': 0.005,
        'velocity_loss_weight': 0.05,
        'vector_loss_weight': 0.05,
        'patience': 5,
        'min_delta': 0.001,
    },
    'data': {
        'train_path': None,
        'val_path': None,
        'test_path': None,
        'max_seq_len': 256,
        'num_workers': 2,
        'num_samples': 2000,
        'num_val_samples': 400,
        'synthetic': True,
    },
    'spatial': {
        'max_spatial_distance': 50.0,
        'temporal_decay': 0.1,
        'use_rotation_equivariance': False,
    },
    'logging': {
        'log_dir': './logs',
        'checkpoint_dir': './checkpoints',
        'save_every': 2,
        'log_every': 20,
        'use_wandb': False,
        'wandb_project': 'spatial-llm',
        'wandb_entity': None,
    },
    'evaluation': {
        'eval_every': 1,
        'metrics': ['perplexity', 'spatial_accuracy', 'temporal_consistency'],
    },
}

os.makedirs('configs', exist_ok=True)
with open('configs/kaggle_config.yaml', 'w') as f:
    yaml.dump(CONFIG, f)

print('Config saved.')
print(f"  Model: {CONFIG['model']['d_model']}d, {CONFIG['model']['n_layers']}L, {CONFIG['model']['n_heads']}H")
print(f"  Training: bs={CONFIG['training']['batch_size']}, lr={CONFIG['training']['learning_rate']}, epochs={CONFIG['training']['num_epochs']}")

## Cell 3: Verify Model

In [ ]:
import torch
import torch.nn as nn
from src.models.spatial_transformer import EmbodiedSpatiotemporalLLM
from src.utils.training_utils import count_parameters, compute_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

mc = CONFIG['model']
model = EmbodiedSpatiotemporalLLM(
    vocab_size=mc['vocab_size'], d_model=mc['d_model'],
    n_layers=mc['n_layers'], n_heads=mc['n_heads'],
    n_reference_frames=mc['n_reference_frames'],
    max_seq_len=mc['max_seq_len'], dropout=mc['dropout'],
    n_schema_types=mc.get('n_schema_types', 8),
    memory_slots=mc.get('memory_slots', 256),
    visual_dim=mc.get('visual_dim', 512),
).to(device)

params = count_parameters(model)
print(f'Parameters: {params["trainable"]:,} trainable / {params["total"]:,} total')
print(f'Model size: ~{params["total"] * 4 / 1e6:.1f} MB (float32)')

# Forward pass test (no_grad for inference check)
test_ids = torch.randint(0, mc['vocab_size'], (2, 32)).to(device)
with torch.no_grad():
    logits, aux = model(test_ids)
print(f'\nForward pass: input {test_ids.shape} -> output {logits.shape}')
print(f'Logits range: [{logits.min():.2f}, {logits.max():.2f}]')
print(f'NaN: {torch.isnan(logits).any().item()}, Inf: {torch.isinf(logits).any().item()}')

# Backward pass test (must run forward OUTSIDE no_grad to build the computation graph)
labels = torch.randint(0, mc['vocab_size'], (2, 32)).to(device)
logits_grad, aux_grad = model(test_ids)
loss, loss_dict = compute_loss(logits_grad, labels)
loss.backward()
max_grad = max(p.grad.abs().max().item() for p in model.parameters() if p.grad is not None)
print(f'Loss: {loss.item():.4f}, Max gradient: {max_grad:.4f}')
print('Model verification PASSED.' if not torch.isnan(logits).any() else 'FAILED: NaN detected!')

del model, logits, logits_grad, aux, aux_grad, test_ids, labels
torch.cuda.empty_cache()

## Cell 4: Generate Synthetic Data

In [ ]:
!python generate_data.py

import json
for split in ['spatial_train', 'spatial_val', 'spatial_test']:
    with open(f'data/processed/{split}.json') as f:
        data = json.load(f)
    print(f'{split}: {len(data)} samples')
    print(f'  Example: {data[0]["context"][:80]}...')
    print(f'  Q: {data[0]["question"]}')
    print(f'  A: {data[0]["answer"]}')
    print()

## Cell 5: Train on Synthetic Data

In [ ]:
!python train.py --config configs/kaggle_config.yaml --device cuda

## Cell 6: Download & Prepare Real Datasets

We download and prepare five spatial reasoning datasets:
1. **bAbI Tasks 17 & 19** (Facebook/Meta) - clean symbolic positional reasoning & path finding
2. **SpartQA** (Allen AI) - controlled synthetic spatial QA
3. **CLEVR** (Meta) - object-centric spatial logic (text-only, no images)
4. **GQA** (Stanford) - real human spatial language from scene graphs
5. **NLVR2** (Cornell) - compositional spatial verification (requires manual download)
6. **ScanQA** (ATR-DBI) - 3D scene spatial QA (requires manual download)

Datasets 1-4 are auto-downloaded. Datasets 5-6 require local paths.

In [ ]:
# Download and prepare real spatial reasoning datasets
# Datasets auto-downloaded: bAbI, SpartQA, CLEVR (no images), GQA
# Optional local paths for: NLVR2, ScanQA

# Set these to local paths if you have them on Kaggle:
SPARTQA_PATH = None   # e.g., '/kaggle/input/spartqa/data'
CLEVR_PATH = None     # e.g., '/kaggle/input/clevr/CLEVR_v1.0'
GQA_PATH = None       # e.g., '/kaggle/input/gqa/questions1.2'
NLVR2_PATH = None     # e.g., '/kaggle/input/nlvr2/data'
SCANQA_PATH = None    # e.g., '/kaggle/input/scanqa/data'

# Build the command
cmd = "python scripts/prepare_real_data.py --synthetic-aug 1000"
if SPARTQA_PATH: cmd += f" --spartqa-path {SPARTQA_PATH}"
if CLEVR_PATH:   cmd += f" --clevr-path {CLEVR_PATH}"
if GQA_PATH:     cmd += f" --gqa-path {GQA_PATH}"
if NLVR2_PATH:   cmd += f" --nlvr2-path {NLVR2_PATH}"
if SCANQA_PATH:  cmd += f" --scanqa-path {SCANQA_PATH}"

print(f"Running: {cmd}\n")
!$cmd

# Show results
import json
from collections import Counter
print("\n--- Dataset Summary ---")
for split in ['real_train', 'real_val', 'real_test']:
    with open(f'data/processed/{split}.json') as f:
        data = json.load(f)
    sources = Counter(s['source'] for s in data)
    print(f'\n{split}: {len(data)} samples')
    for src, cnt in sorted(sources.items(), key=lambda x: -x[1]):
        print(f'  {src}: {cnt} ({cnt/len(data)*100:.1f}%)')
    if data:
        print(f'  Example: {data[0]["context"][:100]}...')

## Cell 7: Train on Real Data

In [ ]:
import yaml
import copy

# Config pointing to real data files
real_config = copy.deepcopy(CONFIG)
real_config['data']['train_path'] = 'data/processed/real_train.json'
real_config['data']['val_path'] = 'data/processed/real_val.json'
real_config['data']['test_path'] = 'data/processed/real_test.json'
real_config['data']['synthetic'] = False
real_config['training']['num_epochs'] = 15
real_config['training']['learning_rate'] = 3e-5
real_config['training']['patience'] = 7
real_config['training']['aux_loss_weight'] = 0.02
real_config['training']['temporal_smoothness_weight'] = 0.01
real_config['training']['velocity_loss_weight'] = 0.1
real_config['training']['vector_loss_weight'] = 0.1

with open('configs/real_data_config.yaml', 'w') as f:
    yaml.dump(real_config, f)

print('Training on real spatial reasoning data...')
!python train.py --config configs/real_data_config.yaml --device cuda

## Cell 8: Evaluation

In [ ]:
import torch
import json
import hashlib
import math
from pathlib import Path
from collections import Counter
from src.models.spatial_transformer import EmbodiedSpatiotemporalLLM
from src.utils.training_utils import count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _deterministic_hash(word, vocab_size):
    """Deterministic word-to-id hash (matches training tokenization)."""
    return int(hashlib.sha256(word.encode('utf-8')).hexdigest(), 16) % vocab_size


# Load model
mc = CONFIG['model']
model = EmbodiedSpatiotemporalLLM(
    vocab_size=mc['vocab_size'], d_model=mc['d_model'],
    n_layers=mc['n_layers'], n_heads=mc['n_heads'],
    n_reference_frames=mc['n_reference_frames'],
    max_seq_len=mc['max_seq_len'], dropout=mc['dropout'],
    n_schema_types=mc.get('n_schema_types', 8),
    memory_slots=mc.get('memory_slots', 256),
    visual_dim=mc.get('visual_dim', 512),
).to(device)

ckpt_path = 'checkpoints/best_model.pt'
if Path(ckpt_path).exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f'Loaded checkpoint (epoch {ckpt["epoch"]}, loss {ckpt["loss"]:.4f})')
else:
    print('No checkpoint found, using fresh model')

model.eval()
params = count_parameters(model)
print(f'Parameters: {params["trainable"]:,}')

# Load test data
test_path = 'data/processed/real_test.json'
if not Path(test_path).exists():
    test_path = 'data/processed/spatial_test.json'

with open(test_path) as f:
    test_data = json.load(f)
print(f'\nTest set: {len(test_data)} samples')

# Forward pass on test samples
n_test = min(200, len(test_data))
total_loss = 0.0
success_count = 0

for i in range(n_test):
    sample = test_data[i]
    text = f"{sample['context']} {sample['question']}"
    words = text.lower().split()[:256]
    ids = [_deterministic_hash(w, mc['vocab_size']) for w in words]
    while len(ids) < 256:
        ids.append(0)
    input_ids = torch.tensor([ids[:256]], dtype=torch.long).to(device)
    attention_mask = (input_ids != 0).float()

    with torch.no_grad():
        logits, aux = model(input_ids, attention_mask=attention_mask)
    if not torch.isnan(logits).any():
        success_count += 1
        labels = input_ids.clone()
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, mc['vocab_size']), labels.view(-1)
        )
        total_loss += loss.item()

avg_loss = total_loss / max(success_count, 1)
perplexity = math.exp(min(avg_loss, 100))

print(f'\n{"=" * 50}')
print(f'RESULTS ({n_test} samples)')
print(f'{"=" * 50}')
print(f'Stable forward passes: {success_count}/{n_test}')
print(f'Average loss: {avg_loss:.4f}')
print(f'Perplexity: {perplexity:.2f}')

# Stats by task type and source
task_types = Counter(s.get('task_type', 'unknown') for s in test_data)
sources = Counter(s.get('source', 'unknown') for s in test_data)

print(f'\nBy task type:')
for t, c in task_types.items():
    print(f'  {t}: {c} ({c/len(test_data)*100:.1f}%)')

print(f'\nBy source:')
for s, c in sources.items():
    print(f'  {s}: {c} ({c/len(test_data)*100:.1f}%)')

print(f'\nAux outputs (sample):')
print(f'  Gain mean: {aux["gain"].mean().item():.3f}')
print(f'  Gate mean: {aux["gate_values"].mean().item():.3f}')

# Save results
results = {
    'n_samples': n_test,
    'stable_passes': success_count,
    'avg_loss': avg_loss,
    'perplexity': perplexity,
    'task_type_distribution': dict(task_types),
    'source_distribution': dict(sources),
    'model_params': params['trainable'],
}
os.makedirs('results', exist_ok=True)
with open('results/evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to results/evaluation_results.json')

## Cell 9: Ablation Study

In [ ]:
import torch
import torch.nn as nn
import time
import json
from src.models.spatial_transformer import EmbodiedSpatiotemporalLLM
from src.utils.data_utils import SpatialReasoningDataset, create_spatial_dataloader
from src.utils.training_utils import compute_loss, get_scheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class VanillaTransformerLLM(nn.Module):
    """Baseline: standard transformer without spatial components."""

    def __init__(self, vocab_size=50257, d_model=256, n_layers=6, n_heads=4,
                 max_seq_len=512, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.dropout = nn.Dropout(dropout)
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.position_embedding.weight, std=0.02)

    def forward(self, input_ids, **kwargs):
        b, s = input_ids.shape
        pos = torch.arange(s, device=input_ids.device).unsqueeze(0)
        x = self.dropout(self.token_embedding(input_ids) + self.position_embedding(pos))
        x = self.transformer(x)
        x = self.norm(x)
        return self.lm_head(x), {}


def quick_train(model, name, n_epochs=5, lr=5e-5, batch_size=8):
    """Train a model for a few epochs and return metrics."""
    model.to(device)
    mc = CONFIG['model']
    dataset = SpatialReasoningDataset(
        synthetic=True, max_seq_len=128, num_samples=500,
        visual_dim=mc.get('visual_dim', 512),
    )
    loader = create_spatial_dataloader(dataset, batch_size=batch_size, num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = get_scheduler(optimizer, 'cosine', len(loader) * n_epochs, warmup_steps=50)

    losses = []
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        steps = 0
        for batch in loader:
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(device)
            ids = batch['input_ids']
            labels = batch['labels']
            optimizer.zero_grad()
            logits, aux = model(
                ids,
                schema_types=batch.get('schema_types'),
                spatial_features=batch.get('spatial_features'),
                temporal_distances=batch.get('temporal_distances'),
                spatial_distances=batch.get('spatial_distances'),
                velocity=batch.get('velocity'),
                visual_features=batch.get('visual_features'),
                attention_mask=batch.get('attention_mask'),
            )
            loss, _ = compute_loss(logits, labels, mask=batch.get('attention_mask'))
            if torch.isnan(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            steps += 1
        avg = epoch_loss / max(steps, 1)
        losses.append(avg)
        print(f'  [{name}] Epoch {epoch+1}: loss={avg:.4f}')

    elapsed = time.time() - t0
    n_params = sum(p.numel() for p in model.parameters())
    return {
        'name': name,
        'final_loss': losses[-1],
        'losses': losses,
        'time': elapsed,
        'params': n_params,
    }


# Run ablations
mc = CONFIG['model']
results = []

print('=== Ablation 1: Vanilla Transformer (no spatial) ===')
vanilla = VanillaTransformerLLM(
    vocab_size=mc['vocab_size'], d_model=mc['d_model'],
    n_layers=mc['n_layers'], n_heads=mc['n_heads'],
    max_seq_len=mc['max_seq_len'], dropout=mc['dropout'],
)
results.append(quick_train(vanilla, 'vanilla'))
del vanilla; torch.cuda.empty_cache()

print('\n=== Ablation 2: Spatial LLM (full architecture) ===')
spatial = EmbodiedSpatiotemporalLLM(
    vocab_size=mc['vocab_size'], d_model=mc['d_model'],
    n_layers=mc['n_layers'], n_heads=mc['n_heads'],
    n_reference_frames=mc['n_reference_frames'],
    max_seq_len=mc['max_seq_len'], dropout=mc['dropout'],
    n_schema_types=mc.get('n_schema_types', 8),
    memory_slots=mc.get('memory_slots', 256),
    visual_dim=mc.get('visual_dim', 512),
)
results.append(quick_train(spatial, 'spatial_llm'))
del spatial; torch.cuda.empty_cache()

print('\n=== Ablation 3: Spatial LLM (smaller, 2 layers) ===')
small = EmbodiedSpatiotemporalLLM(
    vocab_size=mc['vocab_size'], d_model=128,
    n_layers=2, n_heads=2,
    n_reference_frames=2,
    max_seq_len=mc['max_seq_len'], dropout=mc['dropout'],
    n_schema_types=mc.get('n_schema_types', 8),
    memory_slots=64,
    visual_dim=mc.get('visual_dim', 512),
)
results.append(quick_train(small, 'spatial_small'))
del small; torch.cuda.empty_cache()

# Summary
print(f'\n{"=" * 60}')
print(f'{"Model":<20} {"Params":>10} {"Final Loss":>12} {"Time (s)":>10}')
print(f'{"-" * 60}')
for r in results:
    print(f'{r["name"]:<20} {r["params"]:>10,} {r["final_loss"]:>12.4f} {r["time"]:>10.1f}')
print(f'{"=" * 60}')

os.makedirs('results', exist_ok=True)
with open('results/ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved to results/ablation_results.json')

## Cell 10: Summary

In [ ]:
import json
from pathlib import Path

print('=' * 60)
print('SPATIAL LLM - EXPERIMENT SUMMARY')
print('=' * 60)

# Load results if available
for rfile in ['results/evaluation_results.json', 'results/ablation_results.json']:
    if Path(rfile).exists():
        with open(rfile) as f:
            data = json.load(f)
        print(f'\n--- {rfile} ---')
        print(json.dumps(data, indent=2))

print('\n' + '=' * 60)
print('Architecture: Embodied Spatiotemporal LLM')
print('  - Hippocampal core (head direction, grid cells, place cells, path integration)')
print('  - Gain field coordinate transform (ego -> allo)')
print('  - Dual-stream gating mechanism')
print('  - Episodic memory augmentation')
print('  - Standard transformer backbone')
print('  - Velocity and visual vector prediction heads')
print('  - NaN-safe training with gradient clipping')
print('\nDatasets used:')
print('  - bAbI Tasks 17 & 19 (symbolic spatial reasoning)')
print('  - GQA (real human spatial language)')
print('  - NLVR2 (compositional verification)')
print('  - SpartQA (controlled synthetic spatial QA)')
print('  - CLEVR (object-centric spatial logic)')
print('=' * 60)